# Anomaly Detection in Critical Minerals Production

This notebook identifies unusual production events — sudden spikes or drops — in BGS
critical-minerals time series data.  Such anomalies are often signals of real-world
disruptions:

| Pattern | Possible cause |
|---|---|
| Sharp production **drop** | Mine collapse, export ban, sanctions, conflict |
| Sharp production **spike** | New mine opening, statistical revision, reclassification |
| High **volatility** over many years | Political instability, boom-bust commodity cycles |

## Methods used

1. **Z-score on year-over-year (YoY) changes** — flags individual year transitions
   where the percentage change is unusually large relative to the historical norm for
   that commodity-country pair.
2. **Isolation Forest** — an unsupervised ML algorithm that scores entire time-series
   (summarised as feature vectors) as anomalous based on how easy they are to isolate
   in feature space.  Series with high volatility and extreme drops/spikes receive
   lower (more negative) anomaly scores.

## 1. Setup

In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest

DATA_DIR = Path("../data/bgs_data")

## 2. Data Preparation

We load the BGS production CSV, keep only rows where `statistic_type == 'Production'`,
coerce numeric columns, and sort by commodity / country / year so that the
`.shift(1)` used in the YoY calculation is applied within the correct temporal order.

In [ ]:
prod_file = DATA_DIR / "bgs_critical_minerals_production.csv"

raw = pd.read_csv(prod_file, low_memory=False)

# Keep production rows only
df = raw[raw["statistic_type"] == "Production"].copy()

# Coerce numerics
df["year"]     = pd.to_numeric(df["year"],     errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

# Drop rows missing key fields
df = df.dropna(subset=["year", "quantity", "commodity", "country"])
df["year"] = df["year"].astype(int)

# Aggregate duplicates (same commodity / country / year can appear multiple times)
df = (
    df.groupby(["commodity", "country", "year"], as_index=False)["quantity"]
    .sum()
)

# Sort for correct shift() behaviour
df = df.sort_values(["commodity", "country", "year"]).reset_index(drop=True)

print(f"Production records  : {len(df):,}")
print(f"Commodities         : {df['commodity'].nunique()}")
print(f"Countries           : {df['country'].nunique()}")
print(f"Year range          : {df['year'].min()} – {df['year'].max()}")

### 2.1  Year-over-Year Change

For each (commodity, country) series we compute:

```
yoy_change = (quantity_t - quantity_{t-1}) / quantity_{t-1}
```

Rows where the previous year is `NaN` (i.e., the first observation in a series) are
dropped, as are rows where `quantity_{t-1} == 0` (division by zero yields `inf`).

In [ ]:
grp = df.groupby(["commodity", "country"])

df["prev_quantity"] = grp["quantity"].shift(1)
df["yoy_change"]    = (df["quantity"] - df["prev_quantity"]) / df["prev_quantity"]

# Replace inf/-inf with NaN, then drop
df["yoy_change"] = df["yoy_change"].replace([np.inf, -np.inf], np.nan)
df_yoy = df.dropna(subset=["prev_quantity", "yoy_change"]).copy()

print(f"Rows with valid YoY : {len(df_yoy):,}")
print(f"\nYoY change summary:")
print(df_yoy["yoy_change"].describe().round(3).to_string())

## 3. Z-Score Anomaly Detection

Within each (commodity, country) group we standardise the YoY change:

```
z = (yoy_change - mean(yoy_change)) / std(yoy_change)
```

Any observation with `|z| > 2.5` is flagged as a **point anomaly** — a single year
transition that stands out as extreme relative to that series' own history.
Groups with fewer than 3 observations are skipped (insufficient history to compute
a meaningful standard deviation).

In [ ]:
ZSCORE_THRESHOLD = 2.5

def group_zscore(series: pd.Series) -> pd.Series:
    """Return z-scores within a group; NaN if std == 0 or fewer than 3 obs."""
    if len(series) < 3:
        return pd.Series(np.nan, index=series.index)
    mu, sigma = series.mean(), series.std(ddof=1)
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(0.0, index=series.index)
    return (series - mu) / sigma


df_yoy["yoy_zscore"] = (
    df_yoy.groupby(["commodity", "country"])["yoy_change"]
    .transform(group_zscore)
)

df_yoy["is_point_anomaly"] = df_yoy["yoy_zscore"].abs() > ZSCORE_THRESHOLD

n_anomalies = df_yoy["is_point_anomaly"].sum()
print(f"Point anomalies flagged (|z| > {ZSCORE_THRESHOLD}): {n_anomalies:,}")
print(f"As % of all YoY rows : {100 * n_anomalies / len(df_yoy):.1f}%")

# Top 20 by z-score magnitude
top20 = (
    df_yoy[df_yoy["is_point_anomaly"]]
    .assign(abs_zscore=df_yoy["yoy_zscore"].abs())
    .sort_values("abs_zscore", ascending=False)
    .head(20)[["commodity", "country", "year", "quantity",
               "prev_quantity", "yoy_change", "yoy_zscore"]]
    .reset_index(drop=True)
)

top20["yoy_change"]   = top20["yoy_change"].map("{:+.1%}".format)
top20["yoy_zscore"]   = top20["yoy_zscore"].round(2)
top20["quantity"]     = top20["quantity"].map("{:,.0f}".format)
top20["prev_quantity"]= top20["prev_quantity"].map("{:,.0f}".format)

print("\nTop 20 point anomalies by |z-score|:")
print(top20.to_string(index=True))

## 4. Isolation Forest — Series-Level Anomaly Detection

Z-scores flag individual year transitions.  Here we take a complementary view:
which **entire time series** are anomalous?

For every (commodity, country) pair with at least **5 years** of data we extract
five features:

| Feature | Description |
|---|---|
| `mean_yoy` | Average year-over-year change |
| `std_yoy` | Standard deviation of YoY changes |
| `max_drop` | Largest single-year decline (most negative YoY) |
| `max_spike` | Largest single-year increase (most positive YoY) |
| `volatility` | Coefficient of variation of quantity (std / mean) |

An `IsolationForest` with `contamination=0.1` labels the 10% of series that are
hardest to explain as anomalous.

In [ ]:
MIN_YEARS = 5

# Build feature matrix — one row per (commodity, country) series
features_rows = []

for (commodity, country), grp_df in df_yoy.groupby(["commodity", "country"]):
    yoy = grp_df["yoy_change"].dropna()
    qty = grp_df["quantity"].dropna()
    if len(yoy) < MIN_YEARS:
        continue

    qty_mean = qty.mean()
    qty_std  = qty.std(ddof=1)
    volatility = qty_std / qty_mean if qty_mean != 0 else np.nan

    features_rows.append({
        "commodity" : commodity,
        "country"   : country,
        "mean_yoy"  : yoy.mean(),
        "std_yoy"   : yoy.std(ddof=1),
        "max_drop"  : yoy.min(),
        "max_spike" : yoy.max(),
        "volatility": volatility,
        "n_years"   : len(yoy),
    })

feat_df = pd.DataFrame(features_rows).dropna()
print(f"Series with >= {MIN_YEARS} years: {len(feat_df):,}")

feature_cols = ["mean_yoy", "std_yoy", "max_drop", "max_spike", "volatility"]
X = feat_df[feature_cols].values

iso = IsolationForest(contamination=0.1, random_state=42)
feat_df["if_label"]  = iso.fit_predict(X)   # -1 = anomaly, 1 = normal
feat_df["if_score"]  = iso.decision_function(X)  # lower = more anomalous

anomalous_series = (
    feat_df[feat_df["if_label"] == -1]
    .sort_values("volatility", ascending=False)
    .reset_index(drop=True)
)

print(f"Anomalous series (Isolation Forest): {len(anomalous_series):,}")
print("\nTop anomalous series sorted by volatility:")
display_cols = ["commodity", "country", "n_years", "mean_yoy",
                "std_yoy", "max_drop", "max_spike", "volatility", "if_score"]
print(
    anomalous_series[display_cols]
    .head(20)
    .round(3)
    .to_string(index=True)
)

## 5. Anomaly Time-Series Plots

For the **top 6 most volatile anomalous series** (as ranked by Isolation Forest)
we plot:

* A **blue line** for the full production history.
* **Red X markers** at every year flagged as a point anomaly by the Z-score method.

This lets us visually confirm where the statistical signals align with visible
discontinuities in the underlying data.

In [ ]:
TOP_N_SERIES = 6

top_series = anomalous_series.head(TOP_N_SERIES)[["commodity", "country"]].values.tolist()

# Build one subplot grid: 3 columns x 2 rows
from plotly.subplots import make_subplots

n_cols = 3
n_rows = (TOP_N_SERIES + n_cols - 1) // n_cols

subplot_titles = [
    f"{commodity.title()} — {country}"
    for commodity, country in top_series
]

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=subplot_titles,
    vertical_spacing=0.14,
    horizontal_spacing=0.08,
)

for idx, (commodity, country) in enumerate(top_series):
    row = idx // n_cols + 1
    col = idx %  n_cols + 1

    # Full time series
    series_df = df[
        (df["commodity"] == commodity) & (df["country"] == country)
    ].sort_values("year")

    # Anomaly points (from Z-score)
    anom_df = df_yoy[
        (df_yoy["commodity"] == commodity)
        & (df_yoy["country"]   == country)
        & (df_yoy["is_point_anomaly"])
    ]

    show_legend = (idx == 0)

    fig.add_trace(
        go.Scatter(
            x=series_df["year"],
            y=series_df["quantity"],
            mode="lines+markers",
            line=dict(color="steelblue", width=2),
            marker=dict(size=4, color="steelblue"),
            name="Production",
            showlegend=show_legend,
            hovertemplate="%{x}: %{y:,.0f}<extra></extra>",
        ),
        row=row, col=col,
    )

    if len(anom_df) > 0:
        fig.add_trace(
            go.Scatter(
                x=anom_df["year"],
                y=anom_df["quantity"],
                mode="markers",
                marker=dict(symbol="x", size=10, color="crimson", line=dict(width=2)),
                name="Point anomaly",
                showlegend=show_legend,
                hovertemplate="Anomaly %{x}: %{y:,.0f}<extra></extra>",
            ),
            row=row, col=col,
        )

fig.update_layout(
    title_text="Production History with Z-Score Anomalies (top 6 volatile series)",
    height=520 * n_rows,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(gridcolor="#eeeeee")
fig.update_yaxes(gridcolor="#eeeeee")

fig.show()

## 6. Anomaly Count Heatmap

A calendar-style heatmap shows **how many point anomalies were flagged per
(commodity, year)**.  Darker cells indicate years with an unusually high number of
disruption signals across producer countries — which may reflect global events
(e.g., financial crises, pandemics, trade wars) rather than country-specific factors.

In [ ]:
# Count anomalies per (commodity, year)
anom_counts = (
    df_yoy[df_yoy["is_point_anomaly"]]
    .groupby(["commodity", "year"])
    .size()
    .reset_index(name="anomaly_count")
)

# Keep commodities with at least one anomaly; take top 20 by total anomaly count
top_commodities = (
    anom_counts.groupby("commodity")["anomaly_count"]
    .sum()
    .nlargest(20)
    .index.tolist()
)

anom_counts_top = anom_counts[anom_counts["commodity"].isin(top_commodities)]

pivot = (
    anom_counts_top
    .pivot_table(index="commodity", columns="year",
                 values="anomaly_count", fill_value=0)
)

# Sort commodities by total anomaly count descending
pivot = pivot.loc[
    pivot.sum(axis=1).sort_values(ascending=False).index
]

fig_hm = px.imshow(
    pivot,
    color_continuous_scale="YlOrRd",
    labels=dict(x="Year", y="Commodity", color="Anomaly count"),
    title="Anomaly Count Heatmap — Production Disruptions by Commodity & Year",
    aspect="auto",
)

fig_hm.update_layout(
    height=600,
    xaxis_title="Year",
    yaxis_title="Commodity",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=200, r=60, t=80, b=60),
    coloraxis_colorbar=dict(title="# anomalies"),
)

fig_hm.show()

## Summary

* **Z-score method** (Section 3) efficiently surfaces individual year-transitions that
  are statistical outliers within a commodity-country series.  The threshold of 2.5σ
  balances sensitivity against false-positive rate; adjust `ZSCORE_THRESHOLD` as needed.

* **Isolation Forest** (Section 4) provides a holistic series-level view.  High
  volatility and extreme drops/spikes jointly drive the anomaly score, making it
  complementary to the point-wise Z-score approach.

* **Time-series plots** (Section 5) allow visual validation that flagged anomalies
  correspond to genuine discontinuities rather than data artefacts.

* **Heatmap** (Section 6) reveals systemic disruption years — years where many
  commodities show anomalies simultaneously — which may align with known macroeconomic
  or geopolitical shocks.

### Next steps

* Cross-reference flagged anomalies with the `table_notes` / `figure_notes` columns
  to distinguish data-quality artefacts from genuine supply disruptions.
* Enrich the Isolation Forest feature set with trade-flow metrics (export concentration,
  number of active producer countries) from the trade notebooks.
* Apply CUSUM or PELT change-point detection for finer-grained structural break analysis.